# LSTM classificator

Using the dataset `dataset_emails.csv` (or the same dataset you have used in S08_1) create a some text classificators:
* LSTM
* GRU 

Compare the results between LSTM and GRU. Compare the results with the S08_1 methods. 


In [2]:
!pip uninstall urllib3 six

Found existing installation: urllib3 1.25.11
Uninstalling urllib3-1.25.11:
  Would remove:
    /Users/bernardoquindimil/Code/Berniquindimil/NLP_Digital_Portfolio/.conda/lib/python3.12/site-packages/urllib3-1.25.11.dist-info/*
    /Users/bernardoquindimil/Code/Berniquindimil/NLP_Digital_Portfolio/.conda/lib/python3.12/site-packages/urllib3/*
Proceed (Y/n)? ^C


In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [2]:
# Load the dataset, each email of the dataset have a prompt that is the email text and the label that is the type of the email
file_path = "/Users/bernardoquindimil/Code/Berniquindimil/NLP_Digital_Portfolio/S08/dataset_emails.csv"
df = pd.read_csv(file_path)

In [4]:
# Assume dataset has 'text' and 'label' columns
texts = df['prompt'].astype(str).values  # Convert to string in case of NaN
labels = df['label'].values

# Encode labels
label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(labels)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size=0.2, random_state=42)

# Tokenization
max_words = 10000  # Max vocabulary size
max_len = 200  # Max sequence length
tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Padding sequences
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post', truncating='post')

# Build LSTM model
model = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    LSTM(64, return_sequences=True),
    Dropout(0.5),
    LSTM(32),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')  # Assuming binary classification; use softmax for multi-class
])

# Compile model
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train model
model.fit(X_train_pad, y_train, validation_data=(X_test_pad, y_test), epochs=5, batch_size=32)

# Evaluate model
loss, accuracy = model.evaluate(X_test_pad, y_test)
print(f"Test Accuracy: {accuracy:.4f}")


Epoch 1/5


/Users/bernardoquindimil/miniconda3/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


25/25 ━━━━━━━━━━━━━━━━━━━━ 5s 125ms/step - accuracy: 0.1049 - loss: -1.9053 - val_accuracy: 0.0650 - val_loss: -11.6715
Epoch 2/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 116ms/step - accuracy: 0.1094 - loss: -14.0107 - val_accuracy: 0.0650 - val_loss: -26.3910
Epoch 3/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 119ms/step - accuracy: 0.1146 - loss: -27.6165 - val_accuracy: 0.0650 - val_loss: -40.6165
Epoch 4/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 116ms/step - accuracy: 0.0997 - loss: -38.8878 - val_accuracy: 0.0650 - val_loss: -56.4803
Epoch 5/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 117ms/step - accuracy: 0.1118 - loss: -56.9584 - val_accuracy: 0.0650 - val_loss: -74.7635
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.0729 - loss: -76.7988
Test Accuracy: 0.0650
